# Cardiogenesis: avoid-state case study

Evaluation for Section 5, Results 3 (Figure 4c, Table 5). For each start state, with the goal fixed to the FHF state:

1. Run `N_EPISODES` episodes **without** an avoid prompt and find the most visited state on the way to the goal (visits at the first timestep and visits to the goal are not counted).
2. Run `N_EPISODES` episodes **with** that state as the avoid prompt.
3. Compare the fraction of episodes that visit the avoid state and the mean episode length (number of steps, repeated states included).

Run from `raDT/radt` with the `radt` environment active, after training a model with `scripts/train_cardio.sh`. Set `CHECKPOINT` to the checkpoint to evaluate, relative to `CHECKPOINT_DIR` in `raDT/constants.py`.

In [ ]:
CHECKPOINT = 'radt/RUN_NAME/prompt_model_cardiogenesis_iter_ITER'
DEVICE = 'cpu'
N_EPISODES = 200
START_STATES = ['000000000000000', '010111111010011', '000010010100000']
K = 10              # asynchronous Boolean network updates per step (as in training)
MAX_EP_LEN = 30

# model hyperparameters (as in scripts/train_cardio.sh)
N_HEAD, N_LAYER, EMBED_DIM, ADELTA, N_POSITIONS = 6, 6, 384, 1, 2048

In [ ]:
import os
from collections import Counter

import numpy as np
import pandas as pd
import torch
from gymnasium.spaces import Discrete

from prompt_dt.prompt_decision_transformer import ReachAvoidTransformer
from envs.cardiogenesis import CardiogenesisEnv, FHF_STATE, STATE_DIM, PROMPT_DIM, AVOID_REP, to_array, to_readable
from raDT.constants import CHECKPOINT_DIR

model = ReachAvoidTransformer(
    state_dim=STATE_DIM,
    act_dim=15,
    action_space=Discrete(15),
    prompt_dim=PROMPT_DIM,
    max_length=MAX_EP_LEN,
    max_ep_len=MAX_EP_LEN,
    hidden_size=EMBED_DIM,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    n_inner=4 * EMBED_DIM,
    activation_function='relu',
    n_positions=N_POSITIONS,
    resid_pdrop=0.1,
    attn_pdrop=0.1,
    adelta=ADELTA,
    avoid_rep=AVOID_REP,
)
model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, CHECKPOINT), map_location=torch.device(DEVICE)), strict=True)
model.to(device=DEVICE)
model.eval();

In [ ]:
def rollout(env, avoid_state=None):
    # one episode; returns the visited states (including the start state)
    obs, info = env.reset()
    visited = [to_readable(obs["observation"])]

    prompt_goal = torch.from_numpy(obs["desired_goal"].reshape(1, 1, PROMPT_DIM)).to(dtype=torch.float32, device=DEVICE)
    prompt = (prompt_goal, torch.ones((1, 1), device=DEVICE))
    if avoid_state is not None:
        avoid_tokens = torch.from_numpy(to_array(avoid_state).reshape(1, 1, PROMPT_DIM)).to(dtype=torch.float32, device=DEVICE)
        avoid_prompt = (avoid_tokens, torch.ones((1, 1), device=DEVICE))
    else:
        avoid_prompt = None

    states = torch.from_numpy(obs["observation"]).reshape(1, STATE_DIM).to(device=DEVICE, dtype=torch.float32)
    actions = torch.zeros((0,), device=DEVICE, dtype=torch.long)
    timesteps = torch.zeros((1, 1), device=DEVICE, dtype=torch.long)
    for t in range(MAX_EP_LEN):
        actions = torch.cat([actions, torch.zeros((1,), device=DEVICE, dtype=torch.long)], dim=0)
        with torch.no_grad():
            action = model.get_action(states, actions, timesteps, prompt=prompt, avoid_prompt=avoid_prompt,
                                      success_list=torch.tensor([True], device=DEVICE))
        actions[-1] = action
        obs, reward, cost, done, info = env.step(int(action))
        visited.append(to_readable(obs["observation"]))
        states = torch.cat([states, torch.from_numpy(obs["observation"]).reshape(1, STATE_DIM).to(device=DEVICE, dtype=torch.float32)], dim=0)
        timesteps = torch.cat([timesteps, torch.full((1, 1), t + 1, device=DEVICE, dtype=torch.long)], dim=1)
        if info["is_success"]:
            break
    return visited, info["is_success"]


def run(start, avoid_state=None):
    env = CardiogenesisEnv(K, avoids_readable=[], fixed_goal_readable=FHF_STATE, fixed_start_readable=start)
    return [rollout(env, avoid_state) for _ in range(N_EPISODES)]

In [ ]:
results = []
for start in START_STATES:
    without = run(start)
    counts = Counter(s for visited, _ in without for s in visited[1:] if s != FHF_STATE)
    print(f'start {start}: most visited states without an avoid prompt', counts.most_common(3))
    avoid_state = counts.most_common(1)[0][0]

    with_avoid = run(start, avoid_state)
    for condition, episodes in [('- avoid prompt', without), ('+ avoid prompt', with_avoid)]:
        results.append({
            'start': start,
            'avoid state': avoid_state,
            'condition': condition,
            'fraction visited': np.mean([avoid_state in visited for visited, _ in episodes]),
            'mean length': np.mean([len(visited) - 1 for visited, _ in episodes]),
            'success rate': np.mean([success for _, success in episodes]),
        })

pd.DataFrame(results)